In [ ]:
import re
import numpy as np
import pandas as pd

BASE = "/Users/parin/Desktop/prosecutor explorer/worcester"
wdisp_raw = pd.read_parquet(f"{BASE}/charges-disposed.parquet")
wfiled_raw = pd.read_parquet(f"{BASE}/charges-filed.parquet")
print(wdisp_raw.shape, wfiled_raw.shape)

In [ ]:
YEARS = (2017, 2024)
MISSING_MAX = 0.20
MIN_CELL = 30
POP = {y: 862_111 for y in range(2017, 2025)}   # 2020 Census; swap in annual PEP/ACS
EVENTS = {"MA CJ reform act": "2018-04-13", "COVID court closures": "2020-03-16"}
DEF_COL = "def_filled"

def pct(x): return (100 * x).round(1)
def share(s):
    t = s.value_counts(dropna=False)
    return pd.DataFrame({"n": t, "pct": pct(t / t.sum())})
def rate_by(frame, flag, by):
    t = frame.assign(f=np.asarray(flag)).groupby(by, observed=True).f.agg(n="size", k="sum")
    t["pct"] = pct(t.k / t.n)
    return t

In [ ]:
W_DISPO = {
    "DISM": "dismissal", "DWOP": "dismissal", "DWP": "dismissal", "NOL": "dismissal",
    "PCNF": "dismissal", "AS": "dismissal",
    "PTP": "diversion", "DPCC": "diversion", "CATI": "diversion", "VAA": "diversion",
    "CWOF": "cwof", "GPCW": "cwof",
    "GPA": "plea", "GFL": "plea", "GFN": "plea", "SGL": "plea", "GYO": "plea",   # "Guilty" codes assumed plea
    "DPA": "plea", "DFL": "plea", "DFN": "plea",
    "JTG": "trial_conviction", "JWG": "trial_conviction", "SJTL": "trial_conviction",
    "JWNG": "acquittal", "JTNG": "acquittal", "DIRV": "acquittal", "RFNG": "acquittal",
    "NGYO": "acquittal", "NCR": "acquittal",
    "NRSP": "civil", "RFN": "civil", "RFL": "civil", "RSP": "civil",
    "TR": "other", "RCH": "other", "FWOP": "other", "NCF": "other", "PCF": "other",
    "MT": "other", "DPCW": "other",
}
W_LESSER = {"SGL", "SJTL"}
RACE_MAP = {"White": "White", "Black": "Black", "Hispanic": "Hispanic", "Asian": "Asian",
            "American Indian": "Other", "Pacific Islander": "Other", "Middle Eastern": "Other",
            "Other": "Other", "Unknown": "Unknown"}

def prep(d):
    d = d.copy()
    n = len(d); d = d.drop_duplicates(); print(f"dropped {n - len(d):,} dup rows")
    for col in ("charge_description", "disposition_description", "sentence_description"):
        d[col] = (d[col].astype("string").str.replace("Â§", "§", regex=False)
                  .str.replace("Â—", "-", regex=False).str.replace("Â", "", regex=False).str.strip())
    for col in ("filing_date", "disposition_date"):
        d[col] = pd.to_datetime(d[col], errors="coerce")
    bad = (d.disposition_date < "1986-01-01") | (d.disposition_date > pd.Timestamp.today())
    d.loc[bad, "disposition_date"] = pd.NaT

    g = d.groupby("case_id").defendant_id
    n_ids = g.nunique()
    d["def_filled"] = d.defendant_id.fillna(d.case_id.map(g.first().where(n_ids.eq(1))))
    d["case_key"] = d.case_id.astype(str) + "|" + d.def_filled.astype(str)

    d["outcome"] = d.disposition_code.map(W_DISPO)
    d.loc[d.disposition_code.isna(), "outcome"] = "pending"
    print("unmapped codes:", d.loc[d.outcome.isna(), "disposition_code"].value_counts().to_dict())
    d["outcome"] = d.outcome.fillna("other")
    d["lesser"] = d.disposition_code.isin(W_LESSER)

    d["placeholder"] = d.charge_code.astype(str).isin(["666666", "999999"])
    d["offense_cat"] = np.where(d.placeholder, "Other/Unknown", d.offense_group.fillna("Other/Unknown"))
    d["is_civil"] = (d.charge_description.str.contains(r"\s\*(?:\s|$)", na=False)
                     | d.outcome.eq("civil"))
    superior = d.court_name.astype(str).str.contains("Superior", case=False, na=False)
    d["sev_rank"] = np.where(d.is_civil, 0, np.where(superior, 3, 1))   # no statutory class available
    d["sev_class"] = np.select([d.is_civil, superior], ["Civil", "Superior Ct"], "District Ct")

    d["race"] = d.defendant_race.map(RACE_MAP).fillna("Unknown")
    d["sex"] = d.defendant_sex.map({"Male": "Male", "Female": "Female"}).fillna("Unknown")
    d["pend"] = d.outcome.eq("pending")
    print("filing range:", d.filing_date.min(), "to", d.filing_date.max())
    return d

df = prep(wdisp_raw)
dff = prep(wfiled_raw)
print(df.race.value_counts().to_string())

In [ ]:
OUTCOME_RANK = {"trial_conviction": 8, "plea": 7, "cwof": 6, "acquittal": 5, "diversion": 4,
                "dismissal": 3, "declined": 2, "other": 1, "pending": 0}

def build_cases(d):
    lead = (d.sort_values(["case_key", "sev_rank"], ascending=[True, False])
              .drop_duplicates("case_key").set_index("case_key")
              [[DEF_COL, "sev_rank", "sev_class", "offense_cat", "race", "sex", "charge_description", "court_name"]])
    agg = d.groupby("case_key").agg(filed=("filing_date", "min"), dispo=("disposition_date", "max"),
                                    n_charges=("case_key", "size"),
                                    all_civil=("is_civil", "all"), any_pending=("pend", "any"))
    cr = d[~d.is_civil]
    o = (cr.assign(r=cr.outcome.map(OUTCOME_RANK)).sort_values("r", ascending=False)
           .drop_duplicates("case_key").set_index("case_key").outcome)
    c = lead.join(agg).join(o)
    c["outcome"] = c.outcome.fillna("civil")
    c["any_lesser_plea"] = c.index.isin(cr.loc[cr.outcome.eq("plea") & cr.lesser, "case_key"].unique())
    c["year"] = c.filed.dt.year
    dd = (c.dispo - c.filed).dt.days
    c["days_to_dispo"] = dd.where((dd >= 0) & ~c.any_pending)
    return c

cases_all = build_cases(df)
p = cases_all[~cases_all.all_civil & cases_all[DEF_COL].notna()].sort_values("filed")
cases_all["prior_cases"] = p.groupby(DEF_COL).cumcount().reindex(cases_all.index)
cases_all["prior_level"] = (pd.cut(cases_all.prior_cases, [-1, 0, 2, np.inf], labels=["0", "1-2", "3+"])
                              .astype(object).fillna("Unknown"))

cases = cases_all[cases_all.year.between(*YEARS)]
cases_filed = build_cases(dff).pipe(lambda c: c[c.year.between(*YEARS)])   # volume, incl. open cases
ch_w = df[df.filing_date.dt.year.between(*YEARS)]

crim = cases[~cases.all_civil & cases.outcome.ne("pending")]
charged = crim[crim.outcome.ne("declined")]
convicted = charged[charged.outcome.isin(["plea", "trial_conviction"])]
print(len(cases_filed), "filed cases |", len(crim), "criminal disposed |", len(convicted), "convicted")

# cross-check vs raw extract (23,685 / 22,656 / 17,895)
print(cases_filed[~cases_filed.all_civil].groupby("year").size().loc[2017:2019])

In [ ]:
def sent_class(txt):
    if not isinstance(txt, str) or not txt.strip():
        return None
    t = txt.lower()
    if "susp" in t: return "Probation (suspended)"
    if re.search(r"mci|hoc|house of corr|jail|prison|committ|cmtd|dys|\bcom\b", t): return "Incarceration"
    if "prob" in t: return "Probation"
    if re.search(r"fine|cost|restitution|\brest\b|vwf|community serv|\$", t): return "Fine/other"
    if "filed" in t: return "Filed (no sentence)"
    return None

CT_LINE = re.compile(r"^\s*cts?\.?\s*([\d,\s&\-and]+);?(.*)$", re.I)
def split_counts(text):
    by_ct, default = {}, None
    for line in str(text).splitlines():
        m = CT_LINE.match(line)
        if not m:
            default = (default or "") + " " + line
            continue
        nums = set()
        for a, b in re.findall(r"(\d+)\s*-\s*(\d+)", m.group(1)):
            nums.update(range(int(a), int(b) + 1))
        nums.update(int(n) for n in re.findall(r"\d+", re.sub(r"\d+\s*-\s*\d+", "", m.group(1))))
        for n in nums:
            by_ct[n] = m.group(2)
    return by_ct, default

conv_all = df[df.outcome.isin(["plea", "trial_conviction"]) & ~df.is_civil].copy()
cache = {t: split_counts(t) for t in conv_all.sentence_description.dropna().unique()}
def seg(row):
    t = row.sentence_description
    if pd.isna(t): return None
    by_ct, default = cache[t]
    ct = pd.to_numeric(row.charge_count_number, errors="coerce")
    return by_ct.get(int(ct), default) if pd.notna(ct) else default
conv_all["sent_seg"] = conv_all.apply(seg, axis=1)

conv_all["sent"] = conv_all.sent_seg.map(sent_class)
conv_all.loc[conv_all.sent.isna() & conv_all.disposition_code.eq("GFL"), "sent"] = "Filed (no sentence)"
conv_all.loc[conv_all.sent.isna() & conv_all.disposition_code.eq("GFN"), "sent"] = "Fine/other"
conv_all["sent"] = conv_all.sent.fillna("Missing")

UNIT = {"y": 365.25, "m": 30.44, "d": 1, "w": 7}
TERM = re.compile(r"(\d+(?:\.\d+)?)\s*(?:-|to)?\s*(\d+(?:\.\d+)?)?\s*(yrs?|years?|mos?|months?|days?|wks?|weeks?)\b", re.I)
def term_days(s):
    if not isinstance(s, str) or "susp" in s.lower():
        return (np.nan, np.nan)
    m = TERM.search(s)
    if not m:
        return (np.nan, np.nan)
    u = UNIT[m.group(3)[0].lower()]
    lo = float(m.group(1)); hi = float(m.group(2)) if m.group(2) else lo
    return (lo * u, hi * u)
terms = conv_all.sent_seg.map(term_days)
conv_all["term_min_days"] = terms.str[0]
conv_all["term_max_days"] = terms.str[1]

cv_w = conv_all[conv_all.filing_date.dt.year.between(*YEARS)]
sent_missing = cv_w.sent.eq("Missing").mean()
HAS_INCARC = sent_missing <= MISSING_MAX
print(f"sentence missing among convictions: {100 * sent_missing:.1f}% -> HAS_INCARC={HAS_INCARC}")
print(cv_w.sent.value_counts().to_string())

rank = {"Incarceration": 6, "Probation (suspended)": 5, "Probation": 5, "Fine/other": 4,
        "Filed (no sentence)": 2, "Missing": 0}
case_sent = (conv_all.assign(r=conv_all.sent.map(rank)).sort_values("r", ascending=False)
                     .drop_duplicates("case_key").set_index("case_key").sent)

In [ ]:
def sent_class(txt):
    if not isinstance(txt, str) or not txt.strip():
        return None
    t = txt.lower()
    if "susp" in t: return "Probation (suspended)"
    if re.search(r"mci|hoc|house of corr|jail|prison|committ|cmtd|dys|\bcom\b", t): return "Incarceration"
    if "prob" in t: return "Probation"
    if re.search(r"fine|cost|restitution|\brest\b|vwf|community serv|\$", t): return "Fine/other"
    if "filed" in t: return "Filed (no sentence)"
    return None

CT_LINE = re.compile(r"^\s*cts?\.?\s*([\d,\s&\-and]+);?(.*)$", re.I)
def split_counts(text):
    by_ct, default = {}, None
    for line in str(text).splitlines():
        m = CT_LINE.match(line)
        if not m:
            default = (default or "") + " " + line
            continue
        nums = set()
        for a, b in re.findall(r"(\d+)\s*-\s*(\d+)", m.group(1)):
            nums.update(range(int(a), int(b) + 1))
        nums.update(int(n) for n in re.findall(r"\d+", re.sub(r"\d+\s*-\s*\d+", "", m.group(1))))
        for n in nums:
            by_ct[n] = m.group(2)
    return by_ct, default

conv_all = df[df.outcome.isin(["plea", "trial_conviction"]) & ~df.is_civil].copy()
cache = {t: split_counts(t) for t in conv_all.sentence_description.dropna().unique()}
def seg(row):
    t = row.sentence_description
    if pd.isna(t): return None
    by_ct, default = cache[t]
    ct = pd.to_numeric(row.charge_count_number, errors="coerce")
    return by_ct.get(int(ct), default) if pd.notna(ct) else default
conv_all["sent_seg"] = conv_all.apply(seg, axis=1)

conv_all["sent"] = conv_all.sent_seg.map(sent_class)
conv_all.loc[conv_all.sent.isna() & conv_all.disposition_code.eq("GFL"), "sent"] = "Filed (no sentence)"
conv_all.loc[conv_all.sent.isna() & conv_all.disposition_code.eq("GFN"), "sent"] = "Fine/other"
conv_all["sent"] = conv_all.sent.fillna("Missing")

UNIT = {"y": 365.25, "m": 30.44, "d": 1, "w": 7}
TERM = re.compile(r"(\d+(?:\.\d+)?)\s*(?:-|to)?\s*(\d+(?:\.\d+)?)?\s*(yrs?|years?|mos?|months?|days?|wks?|weeks?)\b", re.I)
def term_days(s):
    if not isinstance(s, str) or "susp" in s.lower():
        return (np.nan, np.nan)
    m = TERM.search(s)
    if not m:
        return (np.nan, np.nan)
    u = UNIT[m.group(3)[0].lower()]
    lo = float(m.group(1)); hi = float(m.group(2)) if m.group(2) else lo
    return (lo * u, hi * u)
terms = conv_all.sent_seg.map(term_days)
conv_all["term_min_days"] = terms.str[0]
conv_all["term_max_days"] = terms.str[1]

cv_w = conv_all[conv_all.filing_date.dt.year.between(*YEARS)]
sent_missing = cv_w.sent.eq("Missing").mean()
HAS_INCARC = sent_missing <= MISSING_MAX
print(f"sentence missing among convictions: {100 * sent_missing:.1f}% -> HAS_INCARC={HAS_INCARC}")
print(cv_w.sent.value_counts().to_string())

rank = {"Incarceration": 6, "Probation (suspended)": 5, "Probation": 5, "Fine/other": 4,
        "Filed (no sentence)": 2, "Missing": 0}
case_sent = (conv_all.assign(r=conv_all.sent.map(rank)).sort_values("r", ascending=False)
                     .drop_duplicates("case_key").set_index("case_key").sent)

In [ ]:
yr = ch_w.filing_date.dt.year
inc_w = cv_w[cv_w.sent.eq("Incarceration")]
groups = [
    ({"defendant_id": ch_w[DEF_COL].isna(), "filing_date": ch_w.filing_date.isna(),
      "disposition_date": ch_w.disposition_date.isna(),
      "charge_code": ch_w.charge_code.isna() | ch_w.placeholder,
      "disposition_code": ch_w.disposition_code.isna(),
      "race": ch_w.race.eq("Unknown"), "sex": ch_w.sex.eq("Unknown"),
      "court": ch_w.court_name.isna(), "offense_group": ch_w.offense_cat.eq("Other/Unknown")}, yr),
    ({"sentence_type (convictions)": cv_w.sent.eq("Missing")}, cv_w.filing_date.dt.year),
    ({"sentence_length (incarcerated)": inc_w.term_max_days.isna()}, inc_w.filing_date.dt.year),
]
rows = []
for checks, yrs in groups:
    for k, miss in checks.items():
        by_year = (~miss).groupby(yrs).mean()
        rows.append(dict(field=k, pct_missing=pct(miss.mean()),
                         years_available=",".join(str(int(y)) for y, v in by_year.items() if v > 0),
                         status="ok" if miss.mean() <= MISSING_MAX else "insufficient data"))
rows += [dict(field="severity (felony/misd)", pct_missing=100.0, years_available="",
              status="insufficient data (charge_category 0% populated)"),
         dict(field="dob/age", pct_missing=100.0, years_available="", status="insufficient data (not delivered)"),
         dict(field="bail/detention", pct_missing=100.0, years_available="", status="insufficient data (not delivered)")]
coverage = pd.DataFrame(rows)
coverage

In [ ]:
vol = cases_filed[~cases_filed.all_civil].groupby("year").size().rename("cases").to_frame()
vol["civil_only_excluded"] = cases_filed[cases_filed.all_civil].groupby("year").size()
vol["yoy_pct"] = pct(vol.cases.pct_change())
vol["vs_2017_pct"] = pct(vol.cases / vol.cases.iloc[0] - 1)
vol["per_10k"] = (vol.cases / vol.index.map(POP) * 1e4).round(1)
vol